# 🚀 LightLLM - Kaggle 30-Hour GPU Training Notebook
Train your custom 124M LightLLM Transformer model on Kaggle's Free NVIDIA T4 / P100 GPUs!

In [ ]:
# Step 1: Check GPU availability on Kaggle
!nvidia-smi

In [ ]:
# Step 2: Clone repository & install dependencies
!git clone https://github.com/RABNEER/LightLLM.git
%cd LightLLM
!pip install torch numpy tiktoken tqdm datasets

In [ ]:
# Step 3: Prepare Dataset (with facts, math & stories)
!python prepare_data.py

In [ ]:
# Step 4: Run High-Speed GPU Training (50,000+ Steps)
# Note: Checkpoints save automatically to out/checkpoint.pt every 100 steps!
!python train.py

In [ ]:
# Step 5: Test Chat Inference
import torch
from lightllm.model import LightLLM
from lightllm.config import LightLLMConfig
from lightllm.tokenizer import Tokenizer

config = LightLLMConfig()
model = LightLLM(config)
tokenizer = Tokenizer()
checkpoint = torch.load('out/checkpoint.pt', map_location='cuda')
model.load_state_dict(checkpoint['model'], strict=False)
model.to('cuda').eval()

def chat(prompt):
    formatted = f"User: {prompt}\nAssistant:"
    ids = torch.tensor([tokenizer.encode(formatted)], dtype=torch.long).to('cuda')
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=50, temperature=0.2, top_k=5)
    return tokenizer.decode(out[0].tolist()).split('<|endoftext|>')[0]

print("hello ->", chat("hello"))
print("2+2 ->", chat("2+2"))
print("what is an apple ->", chat("what is an apple"))
print("who created you ->", chat("who created you"))